In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from functions import *

In [ ]:
N = 10000
mu1 = np.array([5.0, 2.0])
mu2 = np.array([7.0, 6.0])
mu3 = np.array([2.0, 8.0])
set_seed(1)

In [ ]:
def generate_class(mu, Sigma, N):

    D1 = np.array(generate_gaussian(mu[0], 1, N))
    D2 = np.array(generate_gaussian(mu[1], 1, N))

    X = np.column_stack((D1, D2))

    X = transform(X, mu, Sigma)

    return X

In [ ]:
def estimate_parameters(X):

    mu_hat = np.mean(X, axis=0)

    X_centered = X - mu_hat

    Sigma_hat = (X_centered.T @ X_centered) / len(X)

    return mu_hat, Sigma_hat

In [ ]:
def confusion_matrix(y_true, y_pred):

    cm = np.zeros((3, 3), dtype=int)

    for i in range(len(y_true)):
        cm[y_true[i] - 1, y_pred[i] - 1] += 1

    return cm

In [ ]:
def run_experiment(Sigma1, Sigma2, Sigma3, title):

    X1 = generate_class(mu1, Sigma1, N)
    X2 = generate_class(mu2, Sigma2, N)
    X3 = generate_class(mu3, Sigma3, N)

    X = np.vstack((X1, X2, X3))

    labels = np.concatenate((np.ones(N),np.ones(N) * 2,np.ones(N) * 3
    )).astype(int)

    index = np.random.permutation(len(X))

    X = X[index]
    labels = labels[index]

    split = int(0.8 * len(X))

    X_train = X[:split]
    y_train = labels[:split]

    X_test = X[split:]
    y_test = labels[split:]
    
    mu_hat = []
    Sigma_hat = []

    for c in [1, 2, 3]:

        Xc = X_train[y_train == c]

        mean_c, covariance_c = estimate_parameters(Xc)

        mu_hat.append(mean_c)
        Sigma_hat.append(covariance_c)

    mu_hat = np.array(mu_hat)
    Sigma_hat = np.array(Sigma_hat)

    scores = np.zeros((len(X_test), 3))

    for c in range(3):

        scores[:, c] = logpdf(
            X_test,
            mu_hat[c],
            Sigma_hat[c]
        )

    y_pred = np.argmax(scores, axis=1) + 1

    accuracy = np.mean(y_pred == y_test) * 100

    cm = confusion_matrix(y_test, y_pred)

    print(title)

    print("\nEstimated Means:")

    print("Class 1:")
    print(mu_hat[0])

    print("Class 2:")
    print(mu_hat[1])

    print("Class 3:")
    print(mu_hat[2])

    print("\nEstimated Covariance Matrices:")

    print("\nClass 1:")
    print(Sigma_hat[0])

    print("\nClass 2:")
    print(Sigma_hat[1])

    print("\nClass 3:")
    print(Sigma_hat[2])

    print("\nAccuracy:", accuracy, "%")

    x_min = -5
    x_max = 14

    y_min = -5
    y_max = 15

    x = np.linspace(x_min, x_max, 300)
    y = np.linspace(y_min, y_max, 300)

    Xg, Yg = np.meshgrid(x, y)

    P = np.column_stack((Xg.ravel(),Yg.ravel()))

    G = np.zeros((len(P), 3))

    for c in range(3):

        G[:, c] = logpdf(P,mu_hat[c],Sigma_hat[c])

    decision = np.argmax(G, axis=1) + 1

    decision = decision.reshape(Xg.shape)

    plt.figure(figsize=(8, 7))

    plt.contour(Xg,Yg,decision,levels=[1.5, 2.5])

    plt.scatter(X_test[y_test == 1, 0],X_test[y_test == 1, 1],
        s=8,alpha=0.25,label="Class 1")

    plt.scatter(X_test[y_test == 2, 0],X_test[y_test == 2, 1],
        s=8,alpha=0.25,label="Class 2")

    plt.scatter(X_test[y_test == 3, 0],X_test[y_test == 3, 1],
        s=8,alpha=0.25,label="Class 3")

    plt.scatter(mu_hat[:, 0],mu_hat[:, 1],marker='x',s=100,
        label="Estimated Means")

    plt.xlim(x_min, x_max)
    plt.ylim(y_min, y_max)

    plt.gca().set_aspect('equal',adjustable='box')

    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.title(title)

    plt.legend()
    plt.grid()
    plt.show()

    return mu_hat, Sigma_hat, cm, accuracy, y_test, scores

In [ ]:
def calculate_roc(y_true, class_scores, positive_class):
    y_binary = (y_true == positive_class).astype(int)

    positives = np.sum(y_binary == 1)
    negatives = np.sum(y_binary == 0)
    if positives == 0 or negatives == 0:
        raise ValueError("ROC requires both positive and negative samples.")

    order = np.argsort(class_scores, kind="mergesort")[::-1]
    sorted_scores = class_scores[order]
    sorted_labels = y_binary[order]

    true_positives = np.cumsum(sorted_labels == 1)
    false_positives = np.cumsum(sorted_labels == 0)

    # Keep only the last sample at each distinct score threshold.
    distinct_indices = np.where(np.diff(sorted_scores) != 0)[0]
    threshold_indices = np.r_[distinct_indices, len(sorted_scores) - 1]

    tp = true_positives[threshold_indices]
    fp = false_positives[threshold_indices]

    # Include the ROC origin (0, 0).
    tpr = np.r_[0.0, tp / positives]
    fpr = np.r_[0.0, fp / negatives]

    # AUC using the trapezoidal rule.
    roc_auc = np.sum(
        (fpr[1:] - fpr[:-1]) * (tpr[1:] + tpr[:-1]) / 2.0
    )
    return fpr, tpr, roc_auc


def plot_roc(y_true, class_scores, title):
    plt.figure(figsize=(7, 7))

    for class_number in [1, 2, 3]:
        fpr, tpr, roc_auc = calculate_roc(
            y_true, class_scores[:, class_number - 1], class_number
        )
        plt.plot(
            fpr, tpr,
            label=f"Class {class_number} vs Rest (AUC = {roc_auc:.3f})"
        )

    plt.plot([0, 1], [0, 1], "k--", label="Random classifier")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.gca().set_aspect("equal", adjustable="box")
    plt.grid(True, alpha=0.3)
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.show()


In [ ]:
Sigma1 = np.array([[2.0, 0.0],[0.0, 2.0]])

Sigma2 = np.array([[2.0, 0.0],[0.0, 2.0]])

Sigma3 = np.array([[2.0, 0.0],[0.0, 2.0]])

mu_hat1, Sigma_hat1, cm1, acc1, y_test1, scores1 = run_experiment(
    Sigma1,Sigma2,Sigma3,"Isotropic Covariance - Same")

plot_roc(y_test1, scores1, "Isotropic Covariance - Same ROC Curve")


In [ ]:
Sigma1 = np.array([[1.0, 0.0],[0.0, 1.0]])

Sigma2 = np.array([[2.0, 0.0],[0.0, 2.0]])

Sigma3 = np.array([[3.0, 0.0],[0.0, 3.0]])

mu_hat2, Sigma_hat2, cm2, acc2, y_test2, scores2 = run_experiment(
    Sigma1,Sigma2,Sigma3,"Isotropic Covariance - Different")

plot_roc(y_test2, scores2, "Isotropic Covariance - Different ROC Curve")


In [ ]:
Sigma1 = np.array([[2.0, 0.0],[0.0, 1.0]])

Sigma2 = np.array([[2.0, 0.0],[0.0, 1.0]])

Sigma3 = np.array([[2.0, 0.0],[0.0, 1.0]])

mu_hat3, Sigma_hat3, cm3, acc3, y_test3, scores3 = run_experiment(
    Sigma1,Sigma2,Sigma3,"Diagonal Covariance - Same")

plot_roc(y_test3, scores3, "Diagonal Covariance - Same ROC Curve")


In [ ]:
Sigma1 = np.array([[2.0, 0.0],[0.0, 1.0]])

Sigma2 = np.array([[3.0, 0.0],[0.0, 2.0]])

Sigma3 = np.array([[1.0, 0.0],[0.0, 3.0]])

mu_hat4, Sigma_hat4, cm4, acc4, y_test4, scores4 = run_experiment(
    Sigma1,Sigma2,Sigma3,"Diagonal Covariance - Different")

plot_roc(y_test4, scores4, "Diagonal Covariance - Different ROC Curve")


In [ ]:
Sigma1 = np.array([[2.0, 0.5],[0.5, 1.5]])

Sigma2 = np.array([[2.0, 0.5],[0.5, 1.5]])

Sigma3 = np.array([[2.0, 0.5],[0.5, 1.5]])

mu_hat5, Sigma_hat5, cm5, acc5, y_test5, scores5 = run_experiment(
    Sigma1,Sigma2,Sigma3,"Full Covariance - Same")

plot_roc(y_test5, scores5, "Full Covariance - Same ROC Curve")


In [ ]:
Sigma1 = np.array([[2.0, 0.5],[0.5, 1.5]])

Sigma2 = np.array([[2.0, -0.8],[-0.8, 2.5]])

Sigma3 = np.array([[1.5, 0.3],[0.3, 2.0]])

mu_hat6, Sigma_hat6, cm6, acc6, y_test6, scores6 = run_experiment(
    Sigma1,Sigma2,Sigma3,"Full Covariance - Different")

plot_roc(y_test6, scores6, "Full Covariance - Different ROC Curve")


In [ ]:
print("Accuracy Comparison")

print("Isotropic - Same      :", acc1, "%")
print("Isotropic - Different :", acc2, "%")
print("Diagonal - Same       :", acc3, "%")
print("Diagonal - Different  :", acc4, "%")
print("Full - Same           :", acc5, "%")
print("Full - Different      :", acc6, "%")

In [ ]:
print("Isotropic - Same")
print(cm1)

print("\nIsotropic - Different")
print(cm2)

print("\nDiagonal - Same")
print(cm3)

print("\nDiagonal - Different")
print(cm4)

print("\nFull - Same")
print(cm5)

print("\nFull - Different")
print(cm6)

In [ ]:
train_data = np.loadtxt("18/trian.txt", delimiter=",")
dev_data = np.loadtxt("18/dev.txt", delimiter=",")

print("Training data shape:", train_data.shape)
print("Development data shape:", dev_data.shape)

In [ ]:
X_train = train_data[:, 0:2]
y_train = train_data[:, 2].astype(int)

X_test = dev_data[:, 0:2]
y_test = dev_data[:, 2].astype(int)

print("Training features:", X_train.shape)
print("Training labels:", y_train.shape)

print("Test features:", X_test.shape)
print("Test labels:", y_test.shape)

In [ ]:
mu_hat = []
Sigma_hat = []

for c in [1, 2, 3]:

    Xc = X_train[y_train == c]

    mu_c = np.mean(Xc, axis=0)

    X_centered = Xc - mu_c

    Sigma_c = (X_centered.T @ X_centered) / len(Xc)

    mu_hat.append(mu_c)
    Sigma_hat.append(Sigma_c)

mu_hat = np.array(mu_hat)
Sigma_hat = np.array(Sigma_hat)

In [ ]:
print("Estimated Means")

for c in range(3):
    print("Class", c + 1, ":", mu_hat[c])

print("\nEstimated Covariance Matrices")

for c in range(3):
    print("\nClass", c + 1)
    print(Sigma_hat[c])

In [ ]:
scores = np.zeros((len(X_test), 3))

for c in range(3):

    scores[:, c] = logpdf(
        X_test,
        mu_hat[c],
        Sigma_hat[c]
    )

y_pred = np.argmax(scores, axis=1) + 1

accuracy = np.mean(y_pred == y_test) * 100

print("Accuracy:", accuracy, "%")

In [ ]:
cm = np.zeros((3, 3), dtype=int)

for i in range(len(y_test)):
    cm[y_test[i] - 1, y_pred[i] - 1] += 1

print("Confusion Matrix")
print(cm)

In [ ]:
all_values = np.concatenate((X_train.ravel(), X_test.ravel()))
plot_min = all_values.min() - 50
plot_max = all_values.max() + 50

x_min = plot_min
x_max = plot_max
y_min = plot_min
y_max = plot_max

x = np.linspace(x_min, x_max, 400)
y = np.linspace(y_min, y_max, 400)

Xg, Yg = np.meshgrid(x, y)
P = np.column_stack((Xg.ravel(), Yg.ravel()))

In [ ]:
G = np.zeros((len(P), 3))

for c in range(3):

    G[:, c] = logpdf(P,mu_hat[c],Sigma_hat[c])

decision = np.argmax(G, axis=1) + 1

decision = decision.reshape(Xg.shape)

In [ ]:
plt.figure(figsize=(8, 7))

plt.contour(Xg,Yg,decision,levels=[1.5, 2.5])

plt.scatter(X_test[y_test == 1, 0],X_test[y_test == 1, 1],
    s=20,alpha=0.5,label="Class 1")

plt.scatter(
    X_test[y_test == 2, 0],
    X_test[y_test == 2, 1],
    s=20,
    alpha=0.5,
    label="Class 2"
)

plt.scatter(
    X_test[y_test == 3, 0],
    X_test[y_test == 3, 1],
    s=20,
    alpha=0.5,
    label="Class 3"
)

plt.scatter(mu_hat[:, 0],mu_hat[:, 1],marker="x",s=120,label="Estimated Means"
)

plt.xlim(x_min, x_max)
plt.ylim(y_min, y_max)

plt.gca().set_aspect("equal",adjustable="box")

plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Real 2-D Data - MLE Decision Boundaries")

plt.legend()
plt.grid()
plt.show()

plot_roc(y_test, scores, "Real Data ROC Curve")


In [ ]:
print("Real Data Results")

print("\nAccuracy:", accuracy, "%")

print("\nConfusion Matrix:")
print(cm)